# Relatório Responsible AI no Azure Machine Learning (dataset NIDDK)

<mark>Este notebook é didático, não deve ser utilizado para realizar diagnósticos e não valida um modelo para uso clínico</mark>

Este notebook usa as versões mais recentes do modelo e dos data assets publicados por `04-azureml-niddk-diabetes.ipynb` para gerar um dashboard Responsible AI no Azure Machine Learning. Ele não treina nem altera o modelo.

O fluxo é intencionalmente compacto: conectar ao workspace, localizar os assets mais recentes, configurar os componentes Responsible AI, enviar um pipeline ao Azure Machine Learning e abrir o dashboard associado ao modelo. Para reprodutibilidade em produção, versões explícitas devem substituir a seleção por `latest`.

## Preparar o Azure Machine Learning local

Execute em um terminal separado, partindo da raiz do repositório, para instalar as dependências e autenticar sua conta Azure:

```powershell
cd ml
uv sync --locked
az login
```

Se tiver mais de uma assinatura, selecione a que contém o workspace:

```powershell
az account set --subscription "<sua-subscription-id>"
```

Crie sua configuração local a partir do exemplo. O arquivo `.env` é ignorado pelo Git e não deve conter senhas, chaves ou tokens:

```powershell
Copy-Item .env.example .env
```

Preencha `AZURE_SUBSCRIPTION_ID`, `AZURE_RESOURCE_GROUP`, `AZURE_ML_WORKSPACE` e `AZURE_ML_COMPUTE`. O compute deve existir no workspace e ser capaz de executar jobs de pipeline.

O notebook usa `DefaultAzureCredential`, que reutiliza a autenticação de `az login`. Ele não cria recursos Azure automaticamente. Antes de enviar o pipeline, a célula de conexão exibe a assinatura, o resource group, o workspace e o compute selecionados.

A geração acontece no Azure Machine Learning usando os componentes Responsible AI mantidos no registro público `azureml`. Ao terminar, o dashboard fica associado à versão do modelo no Studio, na guia **Responsible AI**.

## Importações e caminhos do projeto

### Bibliotecas

O dashboard é montado por componentes do Azure Machine Learning: construtor, análise de erros, explicabilidade e agregação. Não incluímos fairness, causalidade ou contrafactuais: o dataset não documenta atributos sensíveis apropriados e essas análises requerem decisões de domínio e validação adicional.

In [ ]:
from pathlib import Path
import os

from azure.ai.ml import Input, MLClient, dsl
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
import pandas as pd


def find_project_root() -> Path:
    """Retorna a raiz do repositório quando o notebook é executado neste projeto."""
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "ml" / "pyproject.toml").is_file() and (
            candidate / "README.md"
        ).is_file():
            return candidate
    raise FileNotFoundError("Abra este notebook dentro do repositório do projeto.")


def required_environment_variable(name: str) -> str:
    """Lê uma configuração obrigatória do workspace Azure ML."""
    value = os.getenv(name)
    if not value:
        raise RuntimeError(
            f"A variável {name} não está definida. Copie ml/.env.example para ml/.env, "
            "preencha as configurações do seu workspace e execute esta célula novamente."
        )
    return value


def latest_version(assets, asset_kind: str):
    """Retorna a maior versão numérica de assets versionados do Azure ML."""
    versions = list(assets)
    if not versions:
        raise RuntimeError(
            f"Nenhuma versão de {asset_kind} foi encontrada. Execute primeiro "
            "04-azureml-niddk-diabetes.ipynb neste mesmo workspace."
        )
    try:
        return max(versions, key=lambda asset: int(asset.version))
    except ValueError as exc:
        raise RuntimeError(
            f"Há uma versão não numérica em {asset_kind}; selecione uma versão explicitamente."
        ) from exc

### Variáveis

In [ ]:
PROJECT_ROOT = find_project_root()

TARGET_COLUMN = "diabetes"
EXPERIMENT_NAME = "niddk-diabetes-responsible-ai"
REGISTERED_MODEL_NAME = "niddk-diabetes-xgboost"
TRAIN_DATA_ASSET_NAME = "niddk-diabetes-train"
TEST_DATA_ASSET_NAME = "niddk-diabetes-test"
NOTEBOOK_NAME = "05-responsible-ai-niddk-diabetes.ipynb"
MAXIMUM_TEST_ROWS = 5_000

FEATURE_COLUMNS = [
    "gestacoes",
    "glicose_plasmatica",
    "pressao_arterial_diastolica",
    "espessura_triceps",
    "insulina_serica",
    "imc",
    "indice_hereditariedade_diabetes",
    "idade",
]

## Conectar ao Azure Machine Learning

O dashboard é criado e armazenado no workspace configurado pelo aluno. A conexão também cria um cliente para o registro público `azureml`, que hospeda os componentes Responsible AI usados no pipeline.

In [ ]:
load_dotenv(PROJECT_ROOT / "ml" / ".env")

subscription_id = required_environment_variable("AZURE_SUBSCRIPTION_ID")
resource_group = required_environment_variable("AZURE_RESOURCE_GROUP")
workspace_name = required_environment_variable("AZURE_ML_WORKSPACE")
compute_name = required_environment_variable("AZURE_ML_COMPUTE")

credential = DefaultAzureCredential()
ml_client = MLClient(
    credential=credential,
    subscription_id=subscription_id,
    resource_group_name=resource_group,
    workspace_name=workspace_name,
)
ml_client_registry = MLClient(credential=credential, registry_name="azureml")

print("Azure Machine Learning selecionado:")
print(f"- Assinatura: {subscription_id}")
print(f"- Resource group: {resource_group}")
print(f"- Workspace: {workspace_name}")
print(f"- Compute: {compute_name}")

## Selecionar as versões mais recentes do modelo e dos data assets

Para simplificar o exercício, selecionamos a maior versão registrada do modelo e dos data assets de treino e teste. As versões efetivamente escolhidas são impressas, enviadas como tags do job e exibidas no dashboard.

In [ ]:
model = latest_version(
    ml_client.models.list(name=REGISTERED_MODEL_NAME),
    f"modelo {REGISTERED_MODEL_NAME}",
)
train_data_asset = latest_version(
    ml_client.data.list(name=TRAIN_DATA_ASSET_NAME),
    f"data asset {TRAIN_DATA_ASSET_NAME}",
)
test_data_asset = latest_version(
    ml_client.data.list(name=TEST_DATA_ASSET_NAME),
    f"data asset {TEST_DATA_ASSET_NAME}",
)

model_uri = f"azureml:{model.name}:{model.version}"
train_data_uri = f"azureml:{train_data_asset.name}:{train_data_asset.version}"
test_data_uri = f"azureml:{test_data_asset.name}:{test_data_asset.version}"

print(f"Modelo: {model_uri}")
print(f"Data asset de treino: {train_data_uri}")
print(f"Data asset de teste: {test_data_uri}")

In [ ]:
selected_assets = pd.DataFrame(
    [
        {"tipo": "Modelo", "nome": model.name, "versão": model.version, "id": model.id},
        {"tipo": "Treino", "nome": train_data_asset.name, "versão": train_data_asset.version, "id": train_data_asset.id},
        {"tipo": "Teste", "nome": test_data_asset.name, "versão": test_data_asset.version, "id": test_data_asset.id},
    ]
)
selected_assets

In [ ]:
if model.type != AssetTypes.MLFLOW_MODEL:
    raise ValueError(
        f"O modelo selecionado deve ter tipo {AssetTypes.MLFLOW_MODEL}; tipo encontrado: {model.type}."
    )
if train_data_asset.type != AssetTypes.URI_FILE or test_data_asset.type != AssetTypes.URI_FILE:
    raise ValueError("Os data assets de treino e teste devem ser arquivos URI_FILE criados pelo notebook 05.")

print("Assets selecionados são compatíveis com o pipeline Responsible AI.")

## Configurar os componentes Responsible AI

Os componentes são carregados pelo rótulo `latest` do registro `azureml`. O construtor recebe o modelo registrado e os dois data assets. As análises de erros e explicabilidade usam o modelo e o conjunto de teste sem modificar os assets de origem.

In [ ]:
categorical_column_names = "[]"
classes = "[0, 1]"

if TARGET_COLUMN in FEATURE_COLUMNS:
    raise ValueError("A coluna-alvo não pode fazer parte das características.")

print(f"Coluna-alvo: {TARGET_COLUMN}")
print(f"Características ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")
print("Módulos selecionados: análise de erros e explicabilidade.")

## Criar o pipeline Responsible AI

O construtor inicia o dashboard; os componentes de análise de erros e explicabilidade adicionam seus resultados; por fim, o agregador produz o dashboard ligado ao modelo. O compute configurado em `AZURE_ML_COMPUTE` executa todas as etapas no Azure Machine Learning.

In [ ]:
rai_constructor_component = ml_client_registry.components.get(
    name="microsoft_azureml_rai_tabular_insight_constructor", label="latest"
)
rai_erroranalysis_component = ml_client_registry.components.get(
    name="microsoft_azureml_rai_tabular_erroranalysis", label="latest"
)
rai_explanation_component = ml_client_registry.components.get(
    name="microsoft_azureml_rai_tabular_explanation", label="latest"
)
rai_gather_component = ml_client_registry.components.get(
    name="microsoft_azureml_rai_tabular_insight_gather", label="latest"
)

print("Componentes Responsible AI carregados do registro azureml.")

## Gerar e registrar o dashboard

O job é enviado ao experimento `niddk-diabetes-responsible-ai`. Seu nome, URL no Studio e referências aos assets escolhidos são exibidos ao final. O job pode levar alguns minutos, pois o processamento ocorre no compute do workspace.

O dashboard gerado oferece visão geral do modelo, análise de erros e explicabilidade. Ele é uma ferramenta de investigação técnica: os resultados exigem interpretação contextual e não estabelecem causalidade, segurança clínica, validade externa ou adequação para decisões sobre pacientes.

Para análises de fairness, não use `idade` como substituto automático de atributo sensível. Defina atributos sensíveis legítimos e as métricas adequadas apenas após revisão ética, legal e de domínio.

In [ ]:
@dsl.pipeline(
    name="niddk_diabetes_responsible_ai",
    description="Dashboard Responsible AI didático para o modelo NIDDK Diabetes."
)
def responsible_ai_pipeline():
    construct_job = rai_constructor_component(
        title="NIDDK Diabetes - Responsible AI",
        task_type="classification",
        model_input=Input(type=AssetTypes.MLFLOW_MODEL, path=model_uri),
        train_dataset=Input(type=AssetTypes.URI_FILE, path=train_data_uri),
        test_dataset=Input(type=AssetTypes.URI_FILE, path=test_data_uri),
        target_column_name=TARGET_COLUMN,
        categorical_column_names=categorical_column_names,
        maximum_rows_for_test_dataset=MAXIMUM_TEST_ROWS,
        classes=classes,
    )
    error_analysis_job = rai_erroranalysis_component(
        rai_insights_dashboard=construct_job.outputs.rai_insights_dashboard,
        filter_features="[]",
    )
    explanation_job = rai_explanation_component(
        rai_insights_dashboard=construct_job.outputs.rai_insights_dashboard,
        comment="Explicabilidade do modelo XGBoost para fins didáticos; não representa causalidade clínica.",
    )
    gather_job = rai_gather_component(
        constructor=construct_job.outputs.rai_insights_dashboard,
        insight_1=error_analysis_job.outputs.error_analysis,
        insight_2=explanation_job.outputs.explanation,
    )
    return {
        "dashboard": gather_job.outputs.dashboard,
        "ux_json": gather_job.outputs.ux_json,
    }

pipeline_job = responsible_ai_pipeline()
pipeline_job.settings.default_compute = compute_name
pipeline_job.tags = {
    "notebook_name": NOTEBOOK_NAME,
    "model_name": model.name,
    "model_version": str(model.version),
    "train_data_asset_name": train_data_asset.name,
    "train_data_asset_version": str(train_data_asset.version),
    "test_data_asset_name": test_data_asset.name,
    "test_data_asset_version": str(test_data_asset.version),
}
submitted_job = ml_client.jobs.create_or_update(
    pipeline_job,
    experiment_name=EXPERIMENT_NAME,
)

print(f"Job enviado: {submitted_job.name}")
print(f"URL do Studio: {submitted_job.studio_url}")
ml_client.jobs.stream(submitted_job.name)

## Consultar o dashboard registrado

Após a conclusão bem-sucedida, o Azure Machine Learning associa o dashboard à versão selecionada do modelo. No Studio, abra **Models**, selecione o modelo e sua versão, e então use a guia **Responsible AI** para abrir o dashboard.

In [ ]:
completed_job = ml_client.jobs.get(submitted_job.name)
print(f"Job: {completed_job.name}")
print(f"Status: {completed_job.status}")
print(f"Modelo analisado: {model.name}:{model.version}")
print(f"Data asset de treino: {train_data_asset.name}:{train_data_asset.version}")
print(f"Data asset de teste: {test_data_asset.name}:{test_data_asset.version}")
print(f"Studio: {completed_job.studio_url}")

if completed_job.status != "Completed":
    raise RuntimeError(
        "O pipeline Responsible AI não foi concluído. Consulte os logs do job no Azure ML Studio antes de continuar."
    )

## Próximos passos

Explore o dashboard no Studio, começando pela visão geral, pelos coortes de erro e pela explicabilidade. Registre hipóteses e limitações, mas não conclua que o modelo é apto para uso clínico. Em uma versão futura, escolha e documente atributos sensíveis apropriados antes de configurar uma análise de fairness.

In [ ]:
dashboard_output = completed_job.outputs.get("dashboard")
ux_json_output = completed_job.outputs.get("ux_json")

print("Dashboard Responsible AI gerado e associado ao modelo.")
print(f"Saída do dashboard: {dashboard_output}")
print(f"Dados de interface do dashboard: {ux_json_output}")
print("Abra o link do Studio acima e acesse a guia Responsible AI da versão do modelo.")